In [0]:
dbutils.widgets.text("p_environment", "")
v_environment = dbutils.widgets.get("p_environment")

In [0]:
dbutils.widgets.text("p_file_date", "2024-12-30")
v_file_date = dbutils.widgets.get("p_file_date")

In [0]:
%run "../includes/configuration"

In [0]:
%run "../includes/commom_functions"

## Ingestion del archivo "person.json"

###Paso 1 - Leer el archivo JSON usando "DataframeReader" de Spark

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

In [0]:
name_schema = StructType(fields=[
    StructField("forename", StringType(), True),
    StructField("surname", StringType(), True)
])

In [0]:
person_schema = StructType(fields=[
    StructField("personId", IntegerType(), False),
    StructField("personName", name_schema)
])

In [0]:
person_df = spark.read\
    .schema(person_schema)\
    .json(f"{bronze_folder_path}/{v_file_date}/person.json")

### Paso 2 - Renombrar las columnas y añadir nuevas columnas

In [0]:
from pyspark.sql.functions import col, concat, current_timestamp, lit

In [0]:
person_with_colums_df = add_ingestion_date(person_df)\
    .withColumnsRenamed({"personId": "person_id"})\
    .withColumn("environment", lit(v_environment))\
    .withColumn("name", concat(col("personName.forename"), lit(" "), col("personName.surname")))\
    .withColumn("file_date", lit(v_file_date))


### Paso 3 - Eliminar columnas no "requeridas"

In [0]:
person_final_df = person_with_colums_df.drop("personName")

### Paso 4 - Escribir la salida en un formato "Parquet"

In [0]:
#overwrite_partition("movie_silver", "persons", "file_date", v_file_date)

In [0]:
#person_final_df.write.mode("overwrite").parquet(f"{silver_folder_path}/persons")

In [0]:
#person_final_df.write.mode("delta").partitionBy("file_date").format("delta").saveAsTable("movie_silver.persons")
condition_merge = 'tgt.person_id = src.person_id AND tgt.file_date = src.file_date'

incremental_merge("movie_silver", "persons", person_final_df, condition_merge, "file_date")

In [0]:
%sql
SELECT file_date, count(1)
FROM movie_silver.persons
GROUP BY file_date;

file_date,count(1)
2024-12-16,70000
2024-12-23,20000
2024-12-30,14842


In [0]:
dbutils.notebook.exit("Exitoso")